# NYC Yellow Taxi Demand Analysis

**Question:** when and where is yellow-taxi demand highest in New York City, and what does a trip cost?

**Data:** NYC Taxi and Limousine Commission (TLC) Trip Record Data — yellow taxi trip records, published monthly by the TLC. This notebook works on one month at a time. The example file is January 2024 (`yellow_tripdata_2024-01.parquet`). The full dataset goes back to 2009, is updated monthly, and pickup and dropoff location IDs map to 260 NYC taxi zones.

**Tools:** Python, pandas, matplotlib. The SQL version of the same questions is in `sql/taxi_queries.sql`.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

## Load the data

Download the file first — see `data/DATA_SOURCE.md`. One month is enough to start with: January 2024 has 2,964,624 yellow taxi trips.

In [ ]:
FILE = "../data/yellow_tripdata_2024-01.parquet"

df = pd.read_parquet(FILE, engine="pyarrow")
print(df.shape)
df.head()

## Clean the data

TLC data is real and messy. Known problems in these files:

- Negative fares are data-entry errors.
- Zero-distance trips are GPS failures or cancelled rides.
- Trips over 3 hours are almost always a meter left running.
- One published pipeline dropped about 4.5% of rows as bad data.

So I keep only trips with a fare of 0 or more, a total amount of 0 or more, a distance above 0, and a duration between 0 and 3 hours.

In [ ]:
df["tpep_pickup_datetime"] = pd.to_datetime(df["tpep_pickup_datetime"])
df["tpep_dropoff_datetime"] = pd.to_datetime(df["tpep_dropoff_datetime"])
df["trip_duration"] = df["tpep_dropoff_datetime"] - df["tpep_pickup_datetime"]

rows_before = len(df)
clean = df[
    (df["fare_amount"] >= 0)
    & (df["total_amount"] >= 0)
    & (df["trip_distance"] > 0)
    & (df["trip_duration"] >= pd.Timedelta(0))
    & (df["trip_duration"] <= pd.Timedelta(hours=3))
].copy()
rows_after = len(clean)
print("Rows before cleaning:", rows_before)
print("Rows after cleaning:", rows_after)
print("Rows dropped:", rows_before - rows_after)

## How big is a month?

January 2024 alone had 2,964,624 yellow taxi trips. For scale, the whole of calendar year 2024 saw 41,169,638 yellow taxi trips — up 19% on calendar year 2023 — with total farebox revenue of $965,887,028. In 2021, by comparison, farebox revenue was just over $486 million across almost 31 million trips.

In [ ]:
trip_count = len(clean)
total_farebox = clean["fare_amount"].sum()
avg_fare = clean["fare_amount"].mean()
avg_distance = clean["trip_distance"].mean()
avg_duration_min = clean["trip_duration"].dt.total_seconds().mean() / 60

print("Trips in this file:", trip_count)
print("Total farebox:", total_farebox)
print("Average fare:", avg_fare)
print("Average distance (miles):", avg_distance)
print("Average duration (minutes):", avg_duration_min)

## Busiest pickup zones

Pickup location IDs map to 260 NYC taxi zones — the zone names are in the taxi zone lookup file (see `data/DATA_SOURCE.md`). In one published analysis of August 2024, Midtown Center had the most pickups, with 44,892.

In [ ]:
top_zones = clean.groupby("PULocationID").size().sort_values(ascending=False)
top_zones.head()

In [ ]:
top_zones.head().plot(kind="barh")
plt.xlabel("Trips")
plt.ylabel("Pickup zone ID")
plt.title("Busiest pickup zones")
plt.gca().invert_yaxis()
plt.show()

## Peak hours

Grouping trips by pickup hour shows when demand is highest during the day.

In [ ]:
clean["pickup_hour"] = clean["tpep_pickup_datetime"].dt.hour
trips_by_hour = clean.groupby("pickup_hour").size()
trips_by_hour

In [ ]:
trips_by_hour.plot(kind="bar")
plt.xlabel("Pickup hour")
plt.ylabel("Trips")
plt.title("Trips by pickup hour")
plt.show()

## What does a trip cost?

From the August 2024 analysis: trips within Manhattan averaged 2.16 miles, a fare of $14.41, and 13.17 minutes. Trips into Manhattan from outside averaged 4.47 miles, $26.62, and 21.09 minutes. Trips out of Manhattan averaged 3.57 miles, $33.91, and 22.54 minutes. Green taxis were only about 2% of total taxi trips.

The cell below breaks the same question down by payment type for whatever month is loaded.

In [ ]:
clean.groupby("payment_type").agg(
    trips=("fare_amount", "size"),
    avg_fare=("fare_amount", "mean"),
    avg_tip=("tip_amount", "mean"),
)

## What I found

- **2024 was a strong year for yellow taxis:** 41,169,638 trips, up 19% on 2023, and $965,887,028 in farebox revenue. There were 13,587 medallion taxicabs licensed, 10,870 of them active, and average farebox per driver was $48,698.
- **Demand is concentrated:** in the August 2024 analysis, Midtown Center alone had 44,892 pickups — the busiest zone.
- **Short trips cost more per mile:** a within-Manhattan trip averaged 2.16 miles for $14.41, while a trip out of Manhattan averaged 3.57 miles for $33.91 — longer trips, but the fare does not grow in proportion to distance.
- **Green taxis are a small share:** only about 2% of total taxi trips.

## Limitations

- One month is a snapshot. January is not the whole year, and demand changes by season.
- The tip column records card tips only. Cash tips are not recorded.
- The TLC notes the trip data comes from the taxi technology providers, not from the TLC itself.
- Full-year farebox and trip totals come from the TLC annual report, not from adding up one month's file.

**Source:** NYC Taxi and Limousine Commission, TLC Trip Record Data.